## Libraries and Data

In [1]:
# Import Libraries

import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import joblib

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Set File Paths

DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(exist_ok=True)

In [3]:
# Load Dataset

energy_df = pd.read_csv(DATA_DIR / "processed" / "energy_household_with_target_step3.csv")

print("Dataset loaded successfully!")
print("Shape:", energy_df.shape)

energy_df.head()

Dataset loaded successfully!
Shape: (3599, 56)


,hhid,zone,state,lga,urca_cat,adult_count,members_under_18,members_18_30,members_31_55,members_over_55,...,fridge_monthly_kwh,ac_monthly_kwh,washing_machine_monthly_kwh,electric_cooker_monthly_kwh,thermal_appliance_monthly_kwh,air_cooler_monthly_kwh,estimated_monthly_kwh,supply_factor,estimated_monthly_kwh_adjusted,consumption_category
0,d8af8ab5-30ab-4d9f-bfe4-81231dbe5dbf,North Central,Niger,mashegu,<1hr to small city/town+,5,13,4,1,0,...,54.0,0.0,0.0,0.0,18.0,0.0,39.300,0.200000,39.300,High Consumption
1,e8245d5c-8130-4e78-b4b0-1053b7ecbc9b,North Central,Niger,mashegu,<1hr to small city/town+,6,4,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,12.375,0.250000,12.375,Moderate Consumption
2,435c8e27-517a-46b9-af04-48830e086d7a,North West,Kano,garun_malam,<1hr to large city,3,3,2,1,0,...,0.0,0.0,0.0,0.0,18.0,0.0,51.225,0.500000,51.225,Very High Consumption
3,9303fa53-9fd2-41a9-9f0d-9567dbe5168e,North West,Kano,garun_malam,<1hr to large city,3,7,2,1,0,...,54.0,0.0,0.0,0.0,18.0,12.0,157.200,0.500000,157.200,Very High Consumption
4,c62cc5a5-29c5-423b-9543-a7b05bda454b,North West,Kano,garun_malam,<1hr to large city,2,4,1,1,0,...,0.0,0.0,0.0,0.0,18.0,0.0,4.875,0.208333,4.875,Moderate Consumption


In [4]:
# Check Target Column

print("Target column summary:")
print(energy_df['estimated_monthly_kwh'].describe())

print("\nMissing target values:")
print(energy_df['estimated_monthly_kwh'].isnull().sum())

Target column summary:
count    3599.000000
mean       39.859834
std        58.436122
min         0.000000
25%         4.800000
50%        19.162500
75%        49.612500
max       570.600000
Name: estimated_monthly_kwh, dtype: float64

Missing target values:
0


##### Create App-Friendly Feature Columns

In [5]:
# Create App-Friendly Feature Columns
#
# Create average AC usage hours feature
# In the dataset, AC usage hours were estimated during target creation.
# For households with AC, we use 4 hours as the default estimated usage.
# For households without AC, we use 0 hours.
energy_df['average_ac_usage_hours'] = np.where(
    energy_df['ac_count'] > 0,
    4,
    0
)

model_features = [
    'household_size',
    'number_of_rooms',
    'daily_supply_hours',
    'light_bulb_count',
    'fan_count',
    'television_count',
    'fridge_count',
    'ac_count',
    'average_ac_usage_hours'
]

target_column = 'estimated_monthly_kwh'

model_df = energy_df[model_features + [target_column]].copy()

model_df.head()

,household_size,number_of_rooms,daily_supply_hours,light_bulb_count,fan_count,television_count,fridge_count,ac_count,average_ac_usage_hours,estimated_monthly_kwh
0,18,8,4.0,15,8,2,1,0,0,39.300
1,10,9,6.0,12,1,1,0,0,0,12.375
2,6,5,12.0,6,2,1,0,0,0,51.225
3,10,10,12.0,20,6,1,1,0,0,157.200
4,6,3,5.0,3,0,0,0,0,0,4.875


In [6]:
## Clean Modelling Dataset

# Convert all modelling columns to numeric
for col in model_features + [target_column]:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

# Replace infinite values with NaN
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# Drop rows with missing values
model_df = model_df.dropna()

# Remove negative values if any exist
model_df = model_df[model_df[target_column] >= 0]

print("Clean modelling dataset shape:", model_df.shape)

model_df.head()

Clean modelling dataset shape: (3599, 10)


,household_size,number_of_rooms,daily_supply_hours,light_bulb_count,fan_count,television_count,fridge_count,ac_count,average_ac_usage_hours,estimated_monthly_kwh
0,18,8,4.0,15,8,2,1,0,0,39.300
1,10,9,6.0,12,1,1,0,0,0,12.375
2,6,5,12.0,6,2,1,0,0,0,51.225
3,10,10,12.0,20,6,1,1,0,0,157.200
4,6,3,5.0,3,0,0,0,0,0,4.875


In [7]:
# Check Feature Summary

model_df.describe()

,household_size,number_of_rooms,daily_supply_hours,light_bulb_count,fan_count,television_count,fridge_count,ac_count,average_ac_usage_hours,estimated_monthly_kwh
count,3599.000000,3599.000000,3599.000000,3599.000000,3599.000000,3599.000000,3599.000000,3599.000000,3599.000000,3599.000000
mean,7.037788,4.784940,7.067241,6.767435,1.886079,0.818839,0.431787,0.057794,0.231175,39.859834
std,5.048538,3.062783,5.815434,4.332175,1.931375,0.905465,0.495394,0.233386,0.933542,58.436122
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000,3.000000,3.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.800000
50%,6.000000,4.000000,6.000000,6.000000,2.000000,1.000000,0.000000,0.000000,0.000000,19.162500
75%,9.000000,6.000000,10.000000,8.000000,3.000000,1.000000,1.000000,0.000000,0.000000,49.612500
max,57.000000,30.000000,24.000000,43.000000,27.000000,12.000000,1.000000,1.000000,4.000000,570.600000


## Feature Selection

In [8]:
# Define Features and Target

X = model_df[model_features]
y = model_df[target_column]

print("Feature data shape:", X.shape)
print("Target data shape:", y.shape)

Feature data shape: (3599, 9)
Target data shape: (3599,)


In [9]:
# Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2879, 9)
X_test shape: (720, 9)
y_train shape: (2879,)
y_test shape: (720,)


## Train Regression Models

We will compare four models:

Linear Regression
Decision Tree Regressor
Random Forest Regressor
Gradient Boosting Regressor